In [20]:
import pandas as pd
import numpy as np
from shutil import copyfile
import os

In [14]:
def load_cryst_grep(filename):
    
    WIDTHS = [9, 6, 9, 9, 9, 7, 7, 7, 14, 7]
    NAMES = ['pdb', 'record', 'a', 'b', 'c', 'alpha', 'beta', 'gamma', 'sg name', 'copies per cell']
    
    df = pd.read_fwf(filename, header=None, widths=WIDTHS, names=NAMES).dropna(axis=1, how="all").dropna(axis=0, how="any")
    df = df[df["record"] == "CRYST1"]

    return df


def get_closest_row(df, value, column_name="volume"):
    closest_index = (df[column_name] - value).abs().idxmin()
    closest_row = df.loc[closest_index]
    return closest_row

In [5]:
metadata = pd.read_csv("/gpfs/cfel/user/tjlane/mpro/allostery/selected_dataset_archive_2024-08-07/metadata.csv")
metadata.sort_values(by="volume")

,dataset_name,refinement_id,data_reduction_id,crystal_id,run_id,beamline,creator_id,final_pdb_path,mtz_path,refinement_mtz_path,...,rfree,rwork,resolution_cut,a,b,c,alpha,beta,gamma,volume
869,MPro_6439_2_001_003_aligned,60690,24118,MPro_6439_2,1,p11,Xavier/Robin,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2173,0.1870,1.720847,112.502752,52.609673,44.364688,90.0,102.582027,90.0,256.276857
444,l4p08_07_001_003_aligned,59279,12157,MPro_373_0,1,p11,Najeeb/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2023,0.1722,1.759405,112.713619,52.494917,44.515803,90.0,102.753785,90.0,256.896648
726,MPro_5897_3_001_003_aligned,60449,23859,MPro_5897_3,1,p11,Nadine/Robin,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2328,0.2013,2.049832,112.405221,52.832846,44.342517,90.0,102.442331,90.0,257.151473
729,MPro_5901_0_001_003_aligned,64482,37370,MPro_5901_0,1,p11,Hina/Robin,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11010091/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2309,0.2009,1.519000,112.171000,52.877000,44.466000,90.0,102.667000,90.0,257.320514
950,l1p22_16_001_003_aligned,60833,13267,MPro_69_3,1,p11,Najeeb/Arwen,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2209,0.1880,1.710938,112.147141,52.712828,44.630984,90.0,102.741316,90.0,257.343341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392,MPro_2642_0_001_003_aligned,58676,11408,MPro_2642_0,1,p11,Nadine/Huijong,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2430,0.2177,1.740011,114.148054,54.053091,44.643972,90.0,100.793509,90.0,270.582538
418,MPro_3193_0_001_003_aligned,58976,11782,MPro_3193_0,1,p11,Nadine/Christina,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2495,0.2199,1.768733,114.212572,54.018622,44.679788,90.0,100.962239,90.0,270.626694
819,MPro_6278_1_001_003_aligned,60614,24027,MPro_6278_1,1,p11,Faisal/Robin,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2378,0.1997,1.980436,114.707941,53.654772,45.099649,90.0,102.813809,90.0,270.658919
429,MPro_3441_1_001_003_aligned,59120,11958,MPro_3441_1,1,p11,Hina/Christina,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,/asap3/petra3/gpfs/p11/2020/data/11009999/scra...,...,0.2472,0.2292,2.068281,114.225887,53.957421,44.743943,90.0,101.045217,90.0,270.663562


In [11]:
n_structures_to_sample = 5
target_volumes = np.linspace(metadata["volume"].min(), metadata["volume"].max(), n_structures_to_sample)
target_volumes

array([256.27685718, 259.93356778, 263.59027837, 267.24698897,
       270.90369956])

In [24]:
for target_volume in target_volumes:
    pdb_path = get_closest_row(metadata, target_volume)["final_pdb_path"]
    copyfile(pdb_path, f'/gpfs/cfel/user/tjlane/mpro/mprodata/lattice_distributions/pdbs/vol{int(target_volume)}.pdb')